# Offline personal-memory research analysis
This notebook uses the bundled saved scores and audit exports. No model API calls.
It separates the authored middleware comparison from model-only public benchmarks.
Install requirements.txt before running. Read docs/DATA.md for what can be reproduced.

In [ ]:
from pathlib import Path
import json, runpy
import pandas as pd
candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidates if (p / 'data/audit/scores.csv').exists()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook inside the extracted repository, or start Jupyter from its root.')
DATA = ROOT / 'data/audit'
pd.set_option('display.max_columns', 15)
print('Offline inputs found. No model calls will be made.')

## 1. Verify the saved evidence
These checks validate exported calculations, not independent ground truth.

In [ ]:
runpy.run_path(str(ROOT / 'scripts/verify_results.py'), run_name='__main__')

## 2. Middleware accuracy and summary sensitivity
The strict summary metric includes a persistent upstream-failure penalty.

In [ ]:
scores = pd.read_csv(DATA / 'scores.csv')
display(scores.pivot(index='arm', columns='model', values='strict_accuracy'))
display(scores[scores.arm == 'rolling_summary'][['model','n','strict_accuracy','answer_only_accuracy','upstream_summary_failures']])

## 3. State checks and processing cost
Both middleware contexts share the same ingestion trace; do not double-count it.

In [ ]:
states = pd.read_csv(DATA / 'state_checks.csv')
display(states.groupby('model')[['state_equal','proposal_exact','recomputed_incorrect_mutation']].agg(['count','sum']))
costs = pd.read_csv(DATA / 'costs.csv')
display(costs[costs.arm.isin(['full_history','middleware_selected','middleware_visible'])])

## 4. Archived family comparisons and failures
Intervals in paired.csv are archived six-family bootstrap results, not newly recomputed here. Failed-answer examples belong to the authored synthetic workload.

In [ ]:
display(pd.read_csv(DATA / 'paired.csv'))
display(pd.read_csv(DATA / 'categories.csv').head(18))
failures = pd.read_csv(DATA / 'failures.csv')
display(failures[['model','arm','family','id','answer','expected']].head(10))

## 5. Recompute public-benchmark results
Pairs all 82,972 saved score rows. Recomputes 20,000-sample clustered intervals and truncation sensitivities. This does not regrade text or invoke the judge.

In [ ]:
runpy.run_path(str(ROOT / 'scripts/analyze_benchmarks.py'), run_name='__main__')
public = json.loads((ROOT / 'results/benchmark_analysis.json').read_text())
display(pd.DataFrame(public).T)

## 6. Public coverage, truncations and archived scoring diagnostics
The boundary and alternative-F1 sensitivities below were computed in the earlier audit from answer text. They are preserved evidence, not new computations by this notebook.

In [ ]:
rows = pd.read_csv(ROOT / 'data/benchmarks/scores.csv.gz')
display(rows.groupby(['model','suite','status']).size().unstack(fill_value=0))
archived = json.loads((DATA / 'public_reproduced_results.json').read_text())
display(archived['ruler_boundary'])
display(archived['locomo_nonadversarial'])
display(pd.read_csv(ROOT / 'data/benchmarks/all_subgroups.csv'))

## 7. Rebuild figures and inspect acceptance limitations

In [ ]:
runpy.run_path(str(ROOT / 'scripts/build_figures.py'), run_name='__main__')
reports = json.loads((DATA / 'v76_recovered_reports.json').read_text())
report = next(x['data'] for x in reports if x['name'] == 'final_report.json')
print('Supported acceptance cases:', report['supported_strict'], '/', report['supported_cases'])
print('Decision:', report['decision'])
print('Figures and benchmark summaries are in results/.')

## Interpretation
- Storage correctness, answer accuracy and total token use are separate measures.
- All-visible memory succeeds on the authored workload; selected-record memory retains GLM failures.
- Model-only benchmark scores cannot establish a middleware benefit.
- The three public metrics are not interchangeable.
- The earlier failed acceptance case remains a failure.
- No new model evaluations were performed by this notebook.